In [ ]:
# --- PocketLM setup (works in Colab and locally) ---
try:
    import pocketlm  # already installed locally / in CI
except ImportError:
    # Colab: install straight from GitHub. The corpus ships *inside* the package,
    # so there is no repo to clone and no working directory to change.
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "git+https://github.com/ni5h4nt/pocketlm"])
    import pocketlm  # noqa: F811

import os
import torch

DEVICE = "micro" if os.environ.get("POCKETLM_CI") else ("cuda" if torch.cuda.is_available() else "cpu")
CORPUS_PATH = pocketlm.corpus_path()   # packaged data: valid from any directory
print("device:", DEVICE, "(timing is guidance, not a contract)")

# l4.3 MLP feed-forward

After attention mixes information *across* tokens, the MLP processes each token
*independently*: expand to 4x width, a nonlinearity, project back. It is where
much of the model's capacity lives.

In [ ]:
from torch import nn

torch.manual_seed(0)
dim = 8
mlp = nn.Sequential(nn.Linear(dim, 4 * dim), nn.GELU(), nn.Linear(4 * dim, dim))
x = torch.randn(1, 4, dim)
out = mlp(x)
print("MLP preserves shape:", tuple(out.shape))

It is position-wise: the same vector gets the same output wherever it sits, so
the MLP never leaks information between positions, that is attention's job.

In [ ]:
same = torch.randn(dim)
x2 = torch.randn(1, 4, dim)
x2[0, 2] = same
assert tuple(out.shape) == (1, 4, dim)
assert torch.allclose(mlp(x2)[0, 2], mlp(same.view(1, 1, dim))[0, 0], atol=1e-6)